In [1]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent 
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [24]:
from dotenv import load_dotenv
from groq import Groq

import sys
import os
from pathlib import Path

from scripts.utils import load_config

import pandas as pd

import json

In [3]:
load_dotenv()

True

In [41]:
def make_messages(system, user, data_path, text_col):
    data = pd.read_csv(data_path)
    texts = data[text_col].unique().tolist()

    messages = [
        [{"role": "system", "content": system},
         {"role": "user", "content": user.replace("{text}", t)}]
        for t in texts
    ]

    return messages

def get_questions_for_coverage(model, messages, temperature):
    client = Groq()
    results = []

    for m in messages:
        chat_completion = client.chat.completions.create(
            messages=m,
            model=model,
            temperature=temperature
        )
    
        results.append(chat_completion.choices[0].message.content)

        dicts = [json.loads(result) for result in results]
        questions = [d['questions'] for d in dicts]
    
    return questions

In [7]:
config = load_config('configs/llm_as_judge.yaml')

In [8]:
question_generation = config['coverage']['question_generation']
question_generation.keys()

dict_keys(['test_path', 'text_col', 'model', 'temperature', 'system', 'user'])

In [9]:
messages = make_messages(
    system=question_generation['system'],
    user=question_generation['user'],
    data_path=question_generation['test_path'],
    text_col=question_generation['text_col']
)

In [42]:
questions = get_questions_for_coverage(
    model=question_generation['model'],
    messages=messages,
    temperature=question_generation['temperature']
)

In [44]:
questions

[['According to the memo, which communication methods are permitted for office correspondence?',
  'Does the new policy prohibit instant messaging only for internal communications, or does it also apply to external communications?',
  'What disciplinary steps are outlined for employees who continue to use instant messaging after the memo is issued?'],
 ['Where did Person2 get stuck in traffic?',
  'What alternative transportation does Person1 propose to reduce environmental impact?',
  'What decision does Person2 make regarding driving to work?'],
 ['Who are getting divorced?',
  'How long have they been separated?',
  'When is the divorce expected to be final?'],
 ['Who is the birthday person addressed by Person1?',
  'What does Person1 ask Person2 to do?',
  'What does Person1 propose to do after the party?'],
 ['When is the Olympic stadium expected to be finished?',
  "How many seats are there in the stadium's stand?",
  'What restriction is indicated by the sign that Person1 sees?'

In [53]:
texts = pd.read_csv('data/small_test.csv')
texts_unique = texts.drop_duplicates('dialogue')
texts_unique['questions'] = questions

C:\Users\pashtet\AppData\Local\Temp\ipykernel_17916\3514566815.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  texts_unique['questions'] = questions


In [56]:
texts_unique.iloc[0]['questions']

['According to the memo, which communication methods are permitted for office correspondence?',
 'Does the new policy prohibit instant messaging only for internal communications, or does it also apply to external communications?',
 'What disciplinary steps are outlined for employees who continue to use instant messaging after the memo is issued?']

In [59]:
test = pd.read_csv('data/test_clean.csv')
base_llm = pd.read_csv('data/base_llm_results.csv')
finetuned = pd.read_csv('data/finetuned_llm_results.csv')

In [82]:
faithfulness_coverage = test.merge(base_llm[['dialogue', 'output']], how='left', on='dialogue').rename({'output':'base_llm'}, axis=1)

In [84]:
faithfulness_coverage = faithfulness_coverage.merge(finetuned[['dialogue', 'output']], how='left', on='dialogue').rename({'output':'finetuned_llm'}, axis=1)

In [93]:
faithfulness_coverage = faithfulness_coverage.drop(columns='Unnamed: 0')

In [96]:
faithfulness_coverage = faithfulness_coverage.drop_duplicates('dialogue')

In [97]:
def make_faithfulness_coverage_df(test_path, base_llm_path, finetuned_path, result_path):
    test = pd.read_csv(test_path)
    base_llm = pd.read_csv(base_llm_path)
    finetuned = pd.read_csv(finetuned_path)

    faithfulness_coverage = test.merge(base_llm[['dialogue', 'output']], how='left', on='dialogue').rename({'output':'base_llm'}, axis=1)
    faithfulness_coverage = faithfulness_coverage.merge(finetuned[['dialogue', 'output']], how='left', on='dialogue').rename({'output':'finetuned_llm'}, axis=1)
    faithfulness_coverage = faithfulness_coverage.drop(columns='Unnamed: 0')
    faithfulness_coverage = faithfulness_coverage.drop_duplicates('dialogue')
    faithfulness_coverage.to_csv(result_path)

In [98]:
def get_answers_for_coverage(model, answer_prompt, text, questions, temperature):
    client = Groq()
    answers = []

    for q in questions:
        prompt = answer_prompt.replace("{text}", text).replace("{question}", q)
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=model,
            temperature=temperature
        )
        answers.append(chat_completion.choices[0].message.content)

    return answers

In [ ]:
config['coverage']

In [99]:
config = load_config('configs/llm_as_judge.yaml')
data = config['data']
model = config['model']
coverage = config['coverage']

In [ ]:
get_answers_for_coverage(
    model=model['model'],
    answer_prompt=coverage['answer_prompt'],
    text
)